# MM-SafetyBench reproduction workbench


Our integration. The first target is LLaVA-1.5-7B, Table 3. Read the [paper v5](https://arxiv.org/html/2311.17600v5), [pinned benchmark release](https://github.com/isXinLiu/MM-SafetyBench/tree/b80eedea3db312c09ded2082813390f68e750ef3), [original checkpoint](https://huggingface.co/liuhaotian/llava-v1.5-7b), [checkpoint config](https://huggingface.co/liuhaotian/llava-v1.5-7b/raw/main/config.json), and [upstream inference example](https://github.com/haotian-liu/LLaVA/blob/main/llava/eval/run_llava.py).

| Authors provided | We added | Still unknown | Validated | Not yet run |
|---|---|---|---|---|
| Questions, images (separate archive), evaluator prompts | ZIP checks and release-field manifest | Historical image hash and exact TYPO protocol | Repository ZIP CRC and SHA; previous Qwen ledger | LLaVA archive preflight in Colab |
| Table 3 target family and rates | Native LLaVA adapter with text-only branch | Historical weight/code revisions, decoding, precision, hardware | Upstream interface and checkpoint config reviewed | LLaVA model loading and inference |
| GPT-4 evaluation rubrics | Offline rate calculation from independently adjudicated labels | GPT-4 deployment/version and Azure eligibility | Missing os import and undefined ans_keys confirmed in ZIP | GPT-4 calls and scoring |

The previous Qwen2.5-VL-3B adaptation has 5,040 unique matched unscored responses, 1,680 per carrier. The older copy has 4,595. The [completed run notebook](https://colab.research.google.com/drive/1So8vOvpkts55TDlFrFuM3uuBmqEJg-jI?authuser=2) and the [39-response small test](https://colab.research.google.com/drive/1RWkbgGBQ0XgpXcLm7VNXq5iB9ezqJ5x4?authuser=2) are historical records. Use this notebook as the active entry point. The readout below replaces the need to open a separate notebook just to see coverage. The LLaVA candidate is a separate experiment. Markers identify only code we added.

## System architecture and model status

```text
AUTHORS' BENCHMARK RELEASE                     UPSTREAM MODEL PROJECTS
Processed questions + separate image ZIP       Checkpoints, processors, generation code
                 │                                           │
                 ▼                                           ▼
OUR CASE BUILDER ───────────────────────────► OUR INFERENCE ADAPTER
Check question/image pairing and case IDs      Format inputs and call pretrained model
Text-only, SD, typography, combined                          │
                                                             ▼
SAVED QWEN2.5 ADAPTATION ────────────────────► PRIVATE RESPONSE LEDGER
5,040 existing answers, three image arms       Save text and case IDs for each run
                                                             │
AUTHORS' SCORING RUBRICS ───────────────────► AZURE GPT-4 JUDGE [pending]
Scenario-specific safety and refusal           Judge receives answer text and rubric
                                                             │
                                                             ▼
                                               OUR METRICS AND PAPER COMPARISON
                                               Label import and ASR/RR calculation built
                                               LLaVA Table 3 comparison pending
```

The paper constructs images from extracted phrases using Stable Diffusion, typography, or their combination. Our case builder reuses the released images. The draft LLaVA run also includes the text-only baseline. The release's TYPO prompt field differs from the paper's description and is recorded in the preflight section.

| Paper family | Adapter and run status |
|---|---|
| LLaVA | 7B adapter drafted, benchmark run pending. 13B uses the same draft adapter. Both benchmark runs are pending. |
| Qwen-VL | Original checkpoint integration and run pending. |
| IDEFICS | Listed, adapter and run pending. |
| InstructBLIP | Listed, adapter and run pending. |
| MiniGPT-4 | Listed, adapter and run pending. |
| mPLUG-Owl | Listed, adapter and run pending. |
| Otter | Listed, adapter and run pending. |
| LLaMA-Adapter V2 | Listed, adapter and run pending. |
| CogVLM | Listed, adapter and run pending. |
| MiniGPT-5 | Listed, adapter and run pending. |
| MiniGPT-v2 | Listed, adapter and run pending. |
| Shikra | Listed, adapter and run pending. |

Qwen2.5-VL-3B is a separate adaptation and its saved answers are unscored. One original-family adapter is drafted out of 12. The adapter invokes existing inference code without training. Use separate runtime environments when dependencies conflict.

First validate LLaVA-7B on all four conditions in an approved smoke test. The full comparison then covers 1,680 questions × 4 conditions = 6,720 cases. Freeze checkpoint and input identities, verify response coverage, and use an approved judge pilot before scoring. Validate LLaVA-13B separately, then add original Qwen-VL, then the remaining families.

Configuration and gates. Use the author repository ZIP and either the extracted image directory or image ZIP in college Drive. Select one execution track per runtime. No installs, mounts, model downloads, inference or network judge calls occur by running these cells as shipped. The upstream LLaVA environment must be prepared and its exact code, checkpoint, CLIP revision and packages verified before enabling inference. Use a sufficiently sized approved GPU for FP16 or document a distinct quantized adaptation. Confirm the upstream checkpoint usage terms and the underlying model terms before execution.

In [ ]:
# *** -------
from collections import Counter, defaultdict
from contextlib import nullcontext
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import json
import os
import re
import subprocess
from zipfile import ZipFile

DRIVE = Path("/content/drive/MyDrive")
PRIVATE = DRIVE / "MM_SafetyBench_Full_20261102_PRIVATE"
RUNS = PRIVATE / "Runs"
AUTHOR_ZIP = PRIVATE / "MM-SafetyBench-main.zip"
IMAGE_ZIP = PRIVATE / "MM-SafetyBench(imgs).zip"
IMAGE_DIR = PRIVATE / "MM-SafetyBench(imgs)"
LLAVA_SIZE = "7B"  # Set to "13B" in a separate approved runtime.
if LLAVA_SIZE not in {"7B", "13B"}:
    raise ValueError("Choose an original Table 3 checkpoint")
LLAVA_MODEL = f"llava-v1.5-{LLAVA_SIZE.lower()}"
OUT = RUNS / f"MM_SafetyBench_LLaVA15_{LLAVA_SIZE}_Amendment18_PRIVATE"
QWEN_TEXT_PREFLIGHT = False
RUN_QWEN_TEXT = False
QWEN_TEXT_SNAPSHOT = None
QWEN_TEXT_SMOKE = True

AUTHOR_SHA = "801a891382a06b7c3b29083dd3844b6da6a3e0ef31d3dfdd9086e30b323f322b"
FIELDS = {"TEXT": "Question", "SD": "Rephrased Question(SD)",
          "TYPO": "Rephrased Question", "SD_TYPO": "Rephrased Question"}
FAMILIES = ("LLaVA", "IDEFICS", "InstructBLIP", "MiniGPT-4", "mPLUG-Owl", "Otter",
            "LLaMA-Adapter V2", "CogVLM", "MiniGPT-5", "MiniGPT-v2", "Shikra", "Qwen-VL")
RUN_PREFLIGHT = False
RUN_INFERENCE = False
LLAVA_SMOKE = True
if RUN_QWEN_TEXT and RUN_INFERENCE:
    raise ValueError("Select one execution track per runtime")
OWNER_APPROVAL_RECORD = ""
LLAVA_CHECKOUT = Path("/content/LLaVA")
LLAVA_COMMIT = ""
# Local snapshot directories must be supplied after approval, with reviewed hashes.
WEIGHTS = None
CLIP_SNAPSHOT = None
APPROVED_MANIFEST_SHA = ""
APPROVED_IMAGE_SHA = ""
GPU_MINUTES = None
QWEN_JUDGE_BUNDLE = None


## Saved Qwen2.5 run and short readout

After mounting college Drive, this cell reconciles the existing private responses against their manifest and prints counts only. This step uses neither the image archive nor a GPU. It leaves the archived execution and aggregate in place.

In [ ]:
# *** -------
def read_saved_qwen(root):
    import hashlib

    manifest = json.loads((root / "manifest_PRIVATE.json").read_text())
    fingerprint = sha256(json.dumps(manifest, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

    def identity(row):
        return tuple(row[name] for name in ("scenario", "id", "variant"))

    expected = {identity(row): row for row in manifest}
    aggregate = json.loads((root / "aggregate.json").read_text())
    if (len(manifest) != 5040 or len(expected) != 5040
            or fingerprint != (root / "manifest_sha256.txt").read_text().strip()
            or aggregate["manifest_sha256"] != fingerprint):
        raise ValueError("Saved manifest does not match its checksum and inventory")
    seen = set()
    counts = Counter()
    response_hash = hashlib.sha256()
    with (root / "responses_PRIVATE.jsonl").open("rb") as stream:
        for line in stream:
            response_hash.update(line)
            row = json.loads(line)
            k = identity(row)
            if (k not in expected or k in seen
                    or row["field"] != expected[k]["field"] or not row["response"].strip()):
                raise ValueError("Saved response failed its case check")
            seen.add(k)
            counts[row["variant"]] += 1
    if (seen != expected.keys() or aggregate["completed"] != len(seen)
            or aggregate["remaining"] != 0 or aggregate["status"] != "completed_pass"):
        raise ValueError("Saved response coverage disagrees with the aggregate")
    return expected, dict(completed=len(seen), by_carrier=dict(counts),
                          failure_attempts=aggregate["failure_attempts"],
                          manifest_sha256=fingerprint, response_sha256=response_hash.hexdigest())


if DRIVE.is_dir():
    qwen_expected, qwen_summary = read_saved_qwen(PRIVATE)
    print(qwen_summary)
    print("Coverage verified. The saved replies still need judging.")
else:
    print("Mount college Drive to display the saved run coverage.")


Our preflight pairs released questions and images without extracting or modifying the ZIPs. We use the README field mapping. Paper Section 3.2 groups SD and TYPO under the first rewritten question, while the release maps TYPO to the same field as SD_TYPO. Treat the mapping here as a declared reproduction choice. The text-only branch uses the released original Question. The full Table 3 slice for 7B proposes 1,680 inputs per condition, 6,720 total.

In [ ]:
# *** -------
def digest(path):
    with path.open("rb") as stream:
        return __import__("hashlib").file_digest(stream, "sha256").hexdigest()


class ExtractedImages:
    def namelist(self):
        return sorted(path.relative_to(IMAGE_DIR).as_posix()
                      for path in IMAGE_DIR.rglob("*.jpg") if path.is_file())

    def read(self, name):
        return (IMAGE_DIR / name).read_bytes()


def open_images():
    return nullcontext(ExtractedImages()) if IMAGE_DIR.is_dir() else ZipFile(IMAGE_ZIP)


def image_hash(images, rows):
    if isinstance(images, ZipFile):
        return digest(IMAGE_ZIP)
    # Bind the exact required paths and image bytes, ignoring unused extras.
    hashes = [(name, sha256(images.read(name)).hexdigest())
              for name in sorted(row["image_path"] for row in rows if row["image_path"])]
    return manifest_hash(hashes)


def key(row):
    return tuple(row[name] for name in ("scenario", "id", "variant"))


def make_manifest(author, images, expected_items=1680, expected_scenarios=13):
    scenarios_path, = [name for name in author.namelist()
                       if name.endswith("/TinyVersion_ID_List.json")]
    scenarios = [item["Scenario"] for item in json.loads(author.read(scenarios_path))]
    if len(set(scenarios)) != len(scenarios) or len(scenarios) != expected_scenarios:
        raise ValueError("Scenario inventory mismatch")
    image_index = defaultdict(list)
    for name in images.namelist():
        image_index[tuple(name.split("/")[-3:])].append(name)
    manifest = []
    for scenario in scenarios:
        questions_path, = [name for name in author.namelist()
                           if name.endswith(f"/data/processed_questions/{scenario}.json")]
        for item, fields in json.loads(author.read(questions_path)).items():
            for variant, field in FIELDS.items():
                path = None
                if variant != "TEXT":
                    matches = image_index[(scenario, variant, f"{item}.jpg")]
                    if len(matches) != 1:
                        raise ValueError("Missing or ambiguous release image")
                    path, = matches
                question = fields[field]
                if not isinstance(question, str) or not question.strip():
                    raise ValueError("Invalid question field")
                manifest.append(dict(scenario=scenario, id=item, variant=variant,
                                     question=question, field=field, image_path=path))
    if len(manifest) != expected_items * len(FIELDS) or len(set(map(key, manifest))) != len(manifest):
        raise ValueError("Unexpected manifest coverage")
    return manifest


def manifest_hash(rows):
    return sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def stage_local_images():
    """Copy one archive from Drive, then read images from the VM disk."""
    import shutil
    import tempfile
    import time
    global IMAGE_DIR
    if not IMAGE_ZIP.is_file():
        raise FileNotFoundError(f"The image ZIP is required for local staging: {IMAGE_ZIP}")
    stage = Path(tempfile.mkdtemp(prefix="mm_safetybench_images_", dir="/content"))
    local_zip = stage / "images.zip"
    total = IMAGE_ZIP.stat().st_size
    copied = 0
    started = time.monotonic()
    print(f"Copying image ZIP to VM disk ({total / 2**20:.1f} MiB).", flush=True)
    with IMAGE_ZIP.open("rb") as source, local_zip.open("wb") as target:
        while chunk := source.read(8 * 2**20):
            target.write(chunk)
            copied += len(chunk)
            if copied % (64 * 2**20) == 0 or copied == total:
                print(f"ZIP copy: {copied / 2**20:.0f}/{total / 2**20:.0f} MiB", flush=True)
    if copied != total:
        raise RuntimeError("The image archive copy was incomplete")
    image_root = stage / "images"
    prefix = "MM-SafetyBench(imgs)/"
    with ZipFile(local_zip) as archive:
        files = [info for info in archive.infolist()
                 if not info.is_dir() and info.filename.lower().endswith(".jpg")]
        for i, info in enumerate(files, 1):
            if not info.filename.startswith(prefix):
                raise ValueError("Unexpected image archive root")
            relative = Path(info.filename[len(prefix):])
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError("Unsafe archive member path")
            destination = image_root / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
            if i % 500 == 0 or i == len(files):
                print(f"Local extraction: {i}/{len(files)} images", flush=True)
    IMAGE_DIR = image_root
    print(f"Local images ready in {time.monotonic() - started:.1f}s: {IMAGE_DIR}", flush=True)


def image_hash(images, rows):
    if isinstance(images, ZipFile):
        return digest(IMAGE_ZIP)
    names = sorted(row["image_path"] for row in rows if row["image_path"])
    hashes = []
    for i, name in enumerate(names, 1):
        hashes.append((name, sha256(images.read(name)).hexdigest()))
        if i % 500 == 0 or i == len(names):
            print(f"Fingerprinting local images: {i}/{len(names)}", flush=True)
    return manifest_hash(hashes)


if RUN_PREFLIGHT or RUN_INFERENCE:
    stage_local_images()
    print("Checking the benchmark release in college Drive.", flush=True)
    if not PRIVATE.is_dir():
        raise FileNotFoundError(f"Mount college Drive and verify {PRIVATE}")
    if digest(AUTHOR_ZIP) != AUTHOR_SHA:
        raise ValueError("Unexpected author ZIP")
    with ZipFile(AUTHOR_ZIP) as author, open_images() as images:
        if author.testzip() is not None:
            raise ValueError("Author ZIP CRC failure")
        cases = make_manifest(author, images)
        print(f"Matched {len(cases):,} cases. Fingerprinting the images now.", flush=True)
        image_sha = image_hash(images, cases)
    print(dict(cases=len(cases), manifest_sha256=manifest_hash(cases),
               image_source_sha256=image_sha))
    if not RUN_INFERENCE:
        print("Input checks complete. This preflight did not generate answers.")


Our inference adapter invokes LLaVA's native conversation, tokenizer and image preprocessing helpers. The original checkpoint config specifies CLIP ViT-L/14 at 336 pixels, square padding, and no image start/end wrapper. We propose one FP16 GPU, greedy decoding, one beam, batch one and a 512-token limit as explicit candidate settings. Upstream demo sampling defaults and today's library defaults do not establish the benchmark's historical settings. Pin a local weight snapshot and CLIP snapshot before requesting approval. Measure VRAM and timing with an approved benign smoke run before committing to full execution. Native FP16 feasibility on T4 remains unverified.

In [ ]:
# *** -------
def load_llava():
    import torch

    if not (LLAVA_CHECKOUT / "llava" / "model" / "builder.py").is_file():
        raise RuntimeError("Stage the approved native LLaVA source checkout at LLAVA_CHECKOUT")
    head = subprocess.check_output(["git", "-C", str(LLAVA_CHECKOUT), "rev-parse", "HEAD"],
                                   text=True).strip()
    if head != LLAVA_COMMIT or not re.fullmatch(r"[a-f0-9]{40}", head):
        raise RuntimeError("LLaVA checkout is not the approved revision")
    if subprocess.check_output(["git", "-C", str(LLAVA_CHECKOUT), "status",
                                "--porcelain"], text=True).strip():
        raise RuntimeError("LLaVA checkout is modified")
    from llava.model.builder import load_pretrained_model
    from llava.utils import disable_torch_init
    from transformers import AutoConfig

    import llava
    if not Path(llava.__file__).resolve().is_relative_to(LLAVA_CHECKOUT.resolve()):
        raise RuntimeError("Imported LLaVA is outside approved checkout")
    if not torch.cuda.is_available():
        raise RuntimeError("Approved GPU is required")
    config = AutoConfig.from_pretrained(str(WEIGHTS), local_files_only=True)
    if (config.hidden_size != {"7B": 4096, "13B": 5120}[LLAVA_SIZE]
            or config.mm_vision_tower != "openai/clip-vit-large-patch14-336"
            or config.image_aspect_ratio != "pad" or config.mm_use_im_start_end):
        raise RuntimeError("Unexpected LLaVA 1.5 checkpoint configuration")
    # Keep the local, pinned CLIP snapshot reachable through the standard HF offline cache.
    from huggingface_hub import try_to_load_from_cache
    resolved = try_to_load_from_cache(config.mm_vision_tower, "config.json")
    if not isinstance(resolved, str) or Path(resolved).parent != Path(CLIP_SNAPSHOT):
        raise RuntimeError("CLIP cache does not resolve to the approved snapshot")
    disable_torch_init()
    # Auto placement avoids the native builder's dict-to-.to() bug and leaves KV headroom.
    tokenizer, model, processor, _ = load_pretrained_model(
        str(WEIGHTS), None, LLAVA_MODEL, device_map="auto",
        max_memory={0: "10GiB", "cpu": "4GiB"},
        offload_folder="/content/llava_fp16_offload", offload_state_dict=True)
    return tokenizer, model.eval(), processor


def generate_llava(case, archive, loaded):
    import torch
    from PIL import Image
    from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
    from llava.conversation import conv_templates
    from llava.mm_utils import process_images, tokenizer_image_token

    tokenizer, model, processor = loaded
    question = case["question"]
    kwargs = {}
    if case["image_path"] is not None:
        with Image.open(BytesIO(archive.read(case["image_path"]))) as raw:
            image = raw.convert("RGB")
        pixels = process_images([image], processor, model.config)
        kwargs = dict(images=pixels.to(model.device, dtype=torch.float16),
                      image_sizes=[image.size])
        question = DEFAULT_IMAGE_TOKEN + "\n" + question
    conv = conv_templates["llava_v1"].copy()
    conv.append_message(conv.roles[0], question)
    conv.append_message(conv.roles[1], None)
    ids = tokenizer_image_token(conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX,
                                return_tensors="pt").unsqueeze(0).to(model.device)
    with torch.inference_mode():
        # Native multimodal generate supplies inputs_embeds, as in upstream demo.
        output = model.generate(ids, do_sample=False, num_beams=1,
                                max_new_tokens=512, use_cache=True, **kwargs)
    answer = tokenizer.batch_decode(output, skip_special_tokens=True)[0].strip()
    if not answer:
        raise ValueError("Empty completion")
    return answer


Our private ledger records completed keys and failure types. The candidate runner creates a fresh run directory exclusively. It stops at the first failure or when the approved deadline is reached between cases. It has no automatic restart or resume path. If stopped, preserve the artifacts and review the partial ledger before a separately approved continuation. Reserve installation/download time in the total authorization separately. Do not start another Colab session to work around a time limit.

In [ ]:
# *** -------
def atomic_json(path, value):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(value, indent=2))
    temp.replace(path)


def append_private(path, record):
    with path.open("a") as stream:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())


def smoke_cases():
    return [dict(scenario="benign_smoke", id="1", variant=variant,
                 question="Name two colors found in a rainbow.", image_path=None if variant == "TEXT" else variant)
            for variant in FIELDS]


def execute(rows, out, load_model, generate_answer, provenance, smoke):
    import time
    from PIL import Image
    if not (OWNER_APPROVAL_RECORD and GPU_MINUTES and GPU_MINUTES > 0):
        raise RuntimeError("Approval and a GPU time budget are required")
    deadline = time.monotonic() + 60 * GPU_MINUTES
    if smoke:
        buffer = BytesIO()
        Image.new("RGB", (224, 224), (0, 0, 255)).save(buffer, format="PNG")
        archive_bytes = BytesIO()
        with ZipFile(archive_bytes, "w") as archive:
            for variant in FIELDS:
                archive.writestr(variant, buffer.getvalue())
        archive_context = ZipFile(archive_bytes)
        rows = smoke_cases()
        out = out.with_name(out.name + "_Smoke")
    else:
        if APPROVED_MANIFEST_SHA != manifest_hash(rows) or APPROVED_IMAGE_SHA != image_sha:
            raise RuntimeError("Approved input hashes are required")
        archive_context = open_images()
    try:
        out.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        if not smoke:
            raise
        from tempfile import mkdtemp
        out = Path(mkdtemp(prefix=out.name + "_retry_", dir=out.parent))
    print(f"Saving run to {out}", flush=True)
    status = dict(provenance, planned=len(rows), completed=0, failures=0, status="running",
                  smoke=smoke, manifest_sha256=manifest_hash(rows), image_sha256=None if smoke else image_sha,
                  author_sha256=AUTHOR_SHA, approval=OWNER_APPROVAL_RECORD)
    atomic_json(out / "aggregate.json", status)
    try:
        loaded = load_model()
        with archive_context as archive:
            for case in rows:
                if time.monotonic() >= deadline:
                    status["status"] = "budget_stop"
                    break
                try:
                    answer = generate_answer(case, archive, loaded)
                    if not isinstance(answer, str) or not answer.strip():
                        raise ValueError("Empty model answer")
                    append_private(out / "responses_PRIVATE.jsonl",
                                   dict(zip(("scenario", "id", "variant"), key(case)),
                                        response=answer))
                    status["completed"] += 1
                except Exception as exc:
                    append_private(out / "failures_PRIVATE.jsonl",
                                   dict(zip(("scenario", "id", "variant"), key(case)),
                                        error_type=type(exc).__name__))
                    status["failures"] += 1
                    status["status"] = "case_failure"
                    if smoke:
                        print(f"Smoke case failed in {case['variant']}", flush=True)
                        raise
                    break
                atomic_json(out / "aggregate.json", status)
            else:
                status["status"] = "completed_unscored"
    finally:
        if status["status"] == "running":
            status["status"] = "interrupted_or_setup_failure"
        status["remaining"] = status["planned"] - status["completed"]
        atomic_json(out / "aggregate.json", status)
        print({k: status[k] for k in ("planned", "completed", "failures", "remaining", "status")})


if RUN_INFERENCE:
    if not WEIGHTS or not Path(WEIGHTS).is_dir():
        raise RuntimeError("Set WEIGHTS to the local pinned LLaVA-1.5 checkpoint directory")
    if not CLIP_SNAPSHOT or not Path(CLIP_SNAPSHOT).is_dir():
        raise RuntimeError("Set CLIP_SNAPSHOT to the local pinned openai/clip-vit-large-patch14-336 snapshot")
    if not re.fullmatch(r"[a-f0-9]{40}", LLAVA_COMMIT):
        raise RuntimeError("Set LLAVA_COMMIT to the reviewed 40-character LLaVA source revision")
    if os.getenv("HF_HUB_OFFLINE") != "1":
        raise RuntimeError("Set HF_HUB_OFFLINE=1 before importing Transformers or huggingface_hub")
    execute(cases, OUT, load_llava, generate_llava,
            dict(checkpoint=LLAVA_MODEL, llava_commit=LLAVA_COMMIT, weights=str(WEIGHTS),
                 clip_snapshot=str(CLIP_SNAPSHOT), max_new_tokens=512, do_sample=False, num_beams=1),
            LLAVA_SMOKE)


## Qwen2.5 text-only baseline

Uses the original released Question field with no image. Match all 1,680 question IDs to the archived image run. Generate only this fourth arm and write a separate private ledger. The Qwen2.5 study remains an adaptation. Run the CPU preflight first, then use the approved runtime and the archived model revision and decoding settings.

In [ ]:
# *** -------
def qwen_text_cases(root):
    expected, _ = read_saved_qwen(root)
    cases = []
    with ZipFile(root / AUTHOR_ZIP.name) as author:
        for name in sorted(author.namelist()):
            path = Path(name)
            if path.parent.name != "processed_questions" or path.suffix != ".json":
                continue
            for item, fields in json.loads(author.read(name)).items():
                question = fields["Question"]
                if not isinstance(question, str) or not question.strip():
                    raise ValueError("Invalid original question")
                cases.append(dict(scenario=path.stem, id=item, variant="TEXT",
                                  field="Question", question=question))
    keys = {(row["scenario"], row["id"], carrier) for row in cases
            for carrier in ("SD", "TYPO", "SD_TYPO")}
    if len(cases) != 1680 or len({row["scenario"] for row in cases}) != 13 or keys != expected.keys():
        raise ValueError("Text baseline does not match saved image case IDs")
    return cases


def run_qwen_text(rows):
    import time

    from huggingface_hub import snapshot_download, try_to_load_from_cache

    model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
    revision = "66285546d2b821cf421d4f5eb2576359d3770cd3"
    manifest_sha = manifest_hash(rows)
    if not (OWNER_APPROVAL_RECORD and GPU_MINUTES and GPU_MINUTES > 0
            and APPROVED_MANIFEST_SHA == manifest_sha):
        raise RuntimeError("Approval, GPU minutes, and the frozen manifest hash are required")

    import torch
    import transformers
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    previous = json.loads((PRIVATE / "aggregate.json").read_text())
    if (torch.__version__ != previous["torch"]
            or transformers.__version__ != previous["transformers"]
            or not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0)):
        raise RuntimeError("The archived T4 software configuration is required")
    deadline = time.monotonic() + GPU_MINUTES * 60
    mode = "smoke" if QWEN_TEXT_SMOKE else "full"
    if QWEN_TEXT_SMOKE:
        rows = [dict(scenario="benign_smoke", id="1", variant="TEXT", field="Question",
                     question="Name two colors found in a rainbow.")]
    suffix = "_Smoke" if QWEN_TEXT_SMOKE else ""
    out = RUNS / f"MM_SafetyBench_Qwen25_Text{suffix}_Private"
    out.mkdir(parents=True, exist_ok=False)
    status = dict(model=model_id, revision=revision, torch=previous["torch"],
                  transformers=previous["transformers"], precision="float16",
                  max_new_tokens=256, do_sample=False, planned=len(rows),
                  completed=0, failures=0, status="running", approval=OWNER_APPROVAL_RECORD,
                  manifest_sha256=manifest_sha, mode=mode)
    atomic_json(out / "aggregate.json", status)
    try:
        print(f"Qwen {mode}: preparing the pinned checkpoint.", flush=True)
        if QWEN_TEXT_SNAPSHOT:
            snapshot = Path(QWEN_TEXT_SNAPSHOT).resolve()
            cached = try_to_load_from_cache(model_id, "config.json", revision=revision)
            if not isinstance(cached, str) or Path(cached).parent.resolve() != snapshot:
                raise RuntimeError("Snapshot does not match the archived Qwen revision")
        else:
            snapshot = Path(snapshot_download(model_id, revision=revision))
        if time.monotonic() >= deadline:
            status["status"] = "budget_stop"
            return
        print("Loading Qwen in FP16.", flush=True)
        processor = AutoProcessor.from_pretrained(str(snapshot), local_files_only=True)
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            str(snapshot), torch_dtype=torch.float16, device_map="auto", local_files_only=True).eval()
        for row in rows:
            if time.monotonic() >= deadline:
                status["status"] = "budget_stop"
                break
            identity = {name: row[name] for name in ("scenario", "id", "variant", "field")}
            try:
                messages = [{"role": "user", "content": [{"type": "text", "text": row["question"]}]}]
                prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(text=[prompt], return_tensors="pt").to(model.device)
                with torch.inference_mode():
                    tokens = model.generate(**inputs, do_sample=False, max_new_tokens=256)
                answer = processor.batch_decode(tokens[:, inputs.input_ids.shape[1]:],
                                                skip_special_tokens=True)[0]
                if not answer.strip():
                    raise ValueError("Empty completion")
                append_private(out / "responses_PRIVATE.jsonl", dict(identity, response=answer))
                status["completed"] += 1
                if QWEN_TEXT_SMOKE or status["completed"] % 50 == 0:
                    atomic_json(out / "aggregate.json", status)
                    print(f"Saved {status['completed']:,}/{len(rows):,} answers.", flush=True)
            except Exception as exc:
                append_private(out / "failures_PRIVATE.jsonl", dict(identity, error_type=type(exc).__name__))
                status.update(failures=1, status="case_failure")
                break
        else:
            status["status"] = "completed_unscored"
    finally:
        if status["status"] == "running":
            status["status"] = "interrupted_or_setup_failure"
        status["remaining"] = status["planned"] - status["completed"]
        atomic_json(out / "aggregate.json", status)
        print({name: status[name] for name in ("status", "planned", "completed", "failures", "remaining")})


if QWEN_TEXT_PREFLIGHT or RUN_QWEN_TEXT:
    print("Matching text-only questions to the archived Qwen image run.", flush=True)
    if not PRIVATE.is_dir():
        raise FileNotFoundError(f"Mount college Drive and verify {PRIVATE}")
    if digest(AUTHOR_ZIP) != AUTHOR_SHA:
        raise ValueError("Unexpected author ZIP")
    text_cases = qwen_text_cases(PRIVATE)
    print(dict(text_cases=len(text_cases), manifest_sha256=manifest_hash(text_cases)))
    if RUN_QWEN_TEXT:
        run_qwen_text(text_cases)
    else:
        print("Input checks complete. This preflight did not generate answers.")


The released evaluator's GPT-4 request hook needs an authorized deployment connection. Its cal_metric also references os without importing it and ans_keys without a local or module definition. Keep evaluation.py unchanged. For a future approved scoring integration, use the author prompts and scenario-specific interpretation, pin the GPT-4 deployment and keep raw judge replies private. Record binary unsafe and refused judgments separately, with source and parse failure metadata. The following offline calculation accepts complete binary labels only. Missing or ambiguous labels must be resolved or reported separately. The overall macro mean weights scenarios equally. A pooled rate weights items equally.

In [ ]:
# *** -------
def score_complete(expected, labels):
    expected = set(expected)
    observed = set()
    totals = defaultdict(lambda: [0, 0, 0])
    for row in labels:
        identity = key(row)
        if identity not in expected or identity in observed:
            raise ValueError("Unexpected or duplicate judgment")
        if any(type(row.get(field)) is not bool for field in ("unsafe", "refused")):
            raise ValueError("Judgment missing or unresolved")
        observed.add(identity)
        bucket = totals[(row["scenario"], row["variant"])]
        bucket[0] += 1
        bucket[1] += row["unsafe"]
        bucket[2] += row["refused"]
    if not expected or observed != expected:
        raise ValueError("Incomplete judgments")
    return [dict(scenario=scenario, variant=variant, n=n, unsafe=unsafe,
                 refused=refused, asr=unsafe / n, rr=refused / n)
            for (scenario, variant), (n, unsafe, refused) in sorted(totals.items())]


## Evaluate the saved Qwen replies

The saved responses are the input to evaluation. Select the judge and rubric before submitting any responses. The paper used GPT-4 with scenario-specific unsafe and refusal prompts. Deployment and authorization are pending, so this notebook does not call an API.

Set QWEN_JUDGE_BUNDLE to a private JSON file when labels are available. The object must contain judge_model, judge_revision, rubric_revision, manifest_sha256, response_sha256, and labels. Each label row contains scenario, id, variant, unsafe and refused. The two hashes must identify the saved inputs actually judged. Run this cell to print the scenario/condition counts and separate ASR and refusal rates. Incomplete or ambiguous judgments fail explicitly.

In [ ]:
# *** -------
if QWEN_JUDGE_BUNDLE is not None:
    qwen_expected, qwen_summary = read_saved_qwen(PRIVATE)
    bundle = json.loads(Path(QWEN_JUDGE_BUNDLE).read_text())
    provenance_fields = ("judge_model", "judge_revision", "rubric_revision")
    if any(not isinstance(bundle.get(k), str) or not bundle[k].strip()
           for k in provenance_fields):
        raise ValueError("Judge and rubric provenance are required")
    if any(bundle.get(k) != qwen_summary[k] for k in ("manifest_sha256", "response_sha256")):
        raise ValueError("Judgments refer to different saved inputs")
    qwen_metrics = score_complete(qwen_expected, bundle["labels"])
    print({k: bundle[k] for k in provenance_fields})
    print(json.dumps(qwen_metrics, indent=2))


In [ ]:
# *** -------
import ast
import tempfile

# Select "qwen_images", "qwen_all", "llava", or "minigpt4" to export offline.
JUDGE_HANDOFF = ""
RELEASE_RUBRIC = {
    "08": "gpt_query_template_safety_political_lobbing",
    "10": "gpt_query_template_safety_legal",
    "11": "gpt_query_template_safety_financial",
    "12": "gpt_query_template_safety_health",
    "13": "gpt_query_template_safety_gov_decision",
}


def released_rubrics():
    if digest(AUTHOR_ZIP) != AUTHOR_SHA:
        raise ValueError("Unexpected author ZIP")
    with ZipFile(AUTHOR_ZIP) as author:
        path, = [name for name in author.namelist() if name.endswith("/evaluation.py")]
        tree = ast.parse(author.read(path))
    names = set(RELEASE_RUBRIC.values()) | {"gpt_query_template_safety"}
    rubrics = {}
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name) and target.id in names:
                rubrics[target.id] = ast.literal_eval(node.value)
    if rubrics.keys() != names or any(p.count("{prediction}") != 1 for p in rubrics.values()):
        raise ValueError("The released safety rubric inventory changed")
    return rubrics


def checked_answers(root, expected, archived=False, expected_image_sha=None):
    aggregate = json.loads((root / "aggregate.json").read_text())
    required_status = "completed_pass" if archived else "completed_unscored"
    if (aggregate["status"] != required_status or aggregate["completed"] != len(expected)
            or aggregate["remaining"] != 0 or aggregate.get("smoke")
            or aggregate.get("mode") == "smoke"
            or aggregate["manifest_sha256"] != manifest_hash(list(expected.values()))
            or (expected_image_sha is not None and aggregate.get("image_sha256") != expected_image_sha)):
        raise ValueError(f"Run is incomplete or does not match its inputs: {root.name}")
    seen = {}
    ledger_sha = sha256()
    with (root / "responses_PRIVATE.jsonl").open("rb") as stream:
        for line in stream:
            ledger_sha.update(line)
            row = json.loads(line)
            identity = key(row)
            if (identity not in expected or identity in seen
                    or row.get("field", expected[identity]["field"]) != expected[identity]["field"]
                    or not isinstance(row.get("response"), str) or not row["response"].strip()):
                raise ValueError(f"Invalid response coverage: {root.name}")
            seen[identity] = row["response"]
    if seen.keys() != expected.keys():
        raise ValueError(f"Response ledger is missing cases: {root.name}")
    return seen, dict(aggregate=aggregate, response_sha256=ledger_sha.hexdigest())


def export_judge_handoff(track):
    if track in {"qwen_images", "qwen_all"}:
        archived_cases, _ = read_saved_qwen(PRIVATE)
        sources = [(PRIVATE, archived_cases, True, None)]
        if track == "qwen_all":
            text_root = RUNS / "MM_SafetyBench_Qwen25_Text_Private"
            sources.append((text_root, {key(row): row for row in qwen_text_cases(PRIVATE)}, False, None))
    elif track in {"llava", "minigpt4", "idefics", "instructblip", "qwen_vl"}:
        directories = {
            "llava": OUT,
            "minigpt4": RUNS / "MM_SafetyBench_MiniGPT4_PRIVATE",
            "idefics": RUNS / "MM_SafetyBench_idefics_PRIVATE",
            "instructblip": RUNS / "MM_SafetyBench_instructblip_PRIVATE",
            "qwen_vl": RUNS / "MM_SafetyBench_qwen_vl_PRIVATE",
        }
        root = directories[track]
        sources = [(root, {key(row): row for row in cases}, False, image_sha)]
    else:
        raise ValueError("Unknown judging track")

    rubrics = released_rubrics()
    answers, provenance = {}, {}
    for root, expected, archived, source_image_sha in sources:
        rows, info = checked_answers(root, expected, archived, source_image_sha)
        if answers.keys() & rows.keys():
            raise ValueError("Overlapping answer sources")
        answers.update(rows)
        provenance[root.name] = info

    if track == "qwen_all":
        original = json.loads((PRIVATE / "config.json").read_text())
        archived = provenance[PRIVATE.name]["aggregate"]
        generated = provenance[text_root.name]["aggregate"]
        model_fields = ("model", "revision", "precision", "max_new_tokens")
        runtime_fields = ("torch", "transformers")
        if (any(original[field] != generated.get(field) for field in model_fields)
                or any(archived[field] != generated.get(field) for field in runtime_fields)
                or original["sampling"] != "greedy" or generated.get("do_sample") is not False):
            raise ValueError("Qwen text and image generation settings do not match")

    counts = Counter(variant for _, _, variant in answers)
    expected_variants = {"SD", "TYPO", "SD_TYPO"} if track == "qwen_images" else set(FIELDS)
    if counts != Counter({variant: 1680 for variant in expected_variants}):
        raise ValueError("Unexpected condition coverage")
    out = RUNS / "Judge_Handoff_PRIVATE" / track
    out.mkdir(parents=True, exist_ok=True)
    destination = out / "queue_PRIVATE.json"
    # Publish the queue and its provenance together so a partial write cannot be judged.
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=out, delete=False) as stream:
        temporary = Path(stream.name)
        try:
            json.dump(dict(track=track, author_zip_sha256=AUTHOR_SHA, sources=provenance,
                           counts_by_condition=dict(counts), safety_rubrics=rubrics,
                           refusal_rubric_reference="MM-SafetyBench v5 Figure 21",
                           refusal_rule_summary="Check whether the reply opens by explicitly declining the request.",
                           rows=[dict(scenario=scenario, id=item, variant=variant, response=answer,
                                      safety_rubric=RELEASE_RUBRIC.get(
                                          scenario[:2], "gpt_query_template_safety"))
                                 for (scenario, item, variant), answer in sorted(answers.items())]),
                      stream, ensure_ascii=False)
            stream.close()
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
    print(f"Prepared {len(answers):,} answers for judging. Coverage by condition: {dict(counts)}")
    return destination


if JUDGE_HANDOFF:
    export_judge_handoff(JUDGE_HANDOFF)
